In [1]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

sys.path.append(os.environ["GPTNL_DATALOADER"])

EPOCH_1_FOLDERS = {
    "train": os.path.join(os.environ["GPTNL_1_TOKENIZED"], "buckets_split_train"),
    "validate": os.path.join(os.environ["GPTNL_1_TOKENIZED"], "buckets_split_validate"),
}

EPOCH_2_FOLDERS = {
    "train": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_train"),
    "validate": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_validate"),
}

In [2]:
from sampling.functions import count_tokens
import copy


def get_bucket_from_fp(file_path):
    if "code" in file_path:
        return "code"
    elif "dutch_high_quality" in file_path:
        return "dutch_high_quality"
    elif "dutch_low_medium_quality" in file_path:
        return "dutch_low_medium_quality"
    elif "english_high_quality" in file_path:
        return "english_high_quality"
    elif "english_low_medium_quality" in file_path:
        return "english_low_medium_quality"
    elif "new_high_quality" in file_path:
        return "new_high_quality"
    elif "new_low_medium_quality" in file_path:
        return "new_low_medium_quality"
    elif "other_languages" in file_path:
        return "other_languages"
    else:
        raise ValueError(f"Unknown bucket for file path: {file_path}")


def count_tokens_in_folders(folders):
    counts_per_bucket = {
        "total": 0,
        "code": 0,
        "dutch_high_quality": 0,
        "dutch_low_medium_quality": 0,
        "english_high_quality": 0,
        "english_low_medium_quality": 0,
        "new_high_quality": 0,
        "new_low_medium_quality": 0,
        "other_languages": 0,
    }
    counts_per_split = {
        split: copy.deepcopy(counts_per_bucket) for split in folders.keys()
    }
    for split, folder in folders.items():
        if os.path.isdir(folder):
            print(f"Counting tokens in {split} set...")
            count = counts_per_bucket
            for root, dirs, files in os.walk(folder):
                for file in files:
                    if file.endswith(".npy"):
                        file_path = os.path.join(root, file)
                        bucket = get_bucket_from_fp(file_path)
                        # print(f"Processing {file_path}...")
                        count["total"] += count_tokens(file_path)
                        counts_per_bucket[bucket] += count_tokens(file_path)
                        counts_per_split[split]["total"] += count_tokens(file_path)
                        counts_per_split[split][bucket] += count_tokens(file_path)
    return counts_per_bucket, counts_per_split

In [3]:
epoch_1_counts, epoch_1_split_counts = count_tokens_in_folders(EPOCH_1_FOLDERS)
epoch_2_counts, epoch_2_split_counts = count_tokens_in_folders(EPOCH_2_FOLDERS)

Counting tokens in train set...
Counting tokens in validate set...
Counting tokens in train set...
Counting tokens in validate set...


In [4]:
import pandas as pd

# Create a comprehensive dataframe with all counts
data = []

# Add epoch 1 data
for split, counts in epoch_1_split_counts.items():
    for bucket, count in counts.items():
        data.append(
            {"epoch": "epoch_1", "split": split, "bucket": bucket, "token_count": count}
        )

# Add epoch 2 data
for split, counts in epoch_2_split_counts.items():
    for bucket, count in counts.items():
        data.append(
            {"epoch": "epoch_2", "split": split, "bucket": bucket, "token_count": count}
        )

# Create DataFrame
df_overview = pd.DataFrame(data)

# Create a pivot table for better readability
df_pivot = df_overview.pivot_table(
    index=["epoch", "split"], columns="bucket", values="token_count", fill_value=0
)

print("Token counts overview:")
df_pivot

Token counts overview:


bucket                    code  dutch_high_quality  dutch_low_medium_quality  \
epoch   split                                                                  
epoch_1 train     8.393435e+10        1.458467e+10              6.133430e+09   
        validate  1.704255e+09        2.914704e+08              1.264540e+08   
epoch_2 train     2.273092e+11        2.088203e+10              6.133568e+09   
        validate  4.640329e+09        4.187061e+08              1.255695e+08   

bucket            english_high_quality  english_low_medium_quality  \
epoch   split                                                        
epoch_1 train             4.853224e+10                1.552211e+11   
        validate          9.864046e+08                3.169007e+09   
epoch_2 train             4.853165e+10                1.552223e+11   
        validate          9.843609e+08                3.167814e+09   

bucket            new_high_quality  new_low_medium_quality  other_languages  \
epoch   split                                                                 
epoch_1 train         0.000000e+00            0.000000e+00     4.736504e+10   
        validate      0.000000e+00            0.000000e+00     9.652494e+08   
epoch_2 train         2.444355e+10            8.331059e+09     4.737296e+10   
        validate      4.991515e+08            1.699187e+08     9.667039e+08   

bucket                   total  
epoch   split                   
epoch_1 train     3.557708e+11  
        validate  7.242841e+09  
epoch_2 train     5.382263e+11  
        validate  1.097255e+10

In [5]:
# Create a copy of the pivot table for formatting
df_formatted = df_pivot.copy()

# Convert to billions and round to 2 decimal places
df_formatted = df_formatted / 1e9
df_formatted = df_formatted.round(2)

# Add total rows for each epoch (train + validate)
epochs = df_formatted.index.get_level_values("epoch").unique()
for epoch in epochs:
    epoch_data = df_formatted.loc[epoch]
    total_row = epoch_data.sum()
    # Add the total row to the dataframe
    df_formatted.loc[(epoch, "total"), :] = total_row

# Rename columns to nice looking names
column_mapping = {
    "code": "Code",
    "dutch_high_quality": "Dutch (high quality)",
    "dutch_low_medium_quality": "Dutch (low/medium quality)",
    "english_high_quality": "English (high quality)",
    "english_low_medium_quality": "English (low/medium quality)",
    "new_high_quality": "New (high quality)",
    "new_low_medium_quality": "New (low/medium quality)",
    "other_languages": "Other languages",
    "total": "Total",
}

df_formatted = df_formatted.rename(columns=column_mapping)

print("Token counts (in billions):")
df_formatted = df_formatted.transpose()

df_formatted

Token counts (in billions):


epoch                        epoch_1          epoch_2          epoch_1 epoch_2
split                          train validate   train validate   total   total
bucket                                                                        
Code                           83.93     1.70  227.31     4.64   85.63  231.95
Dutch (high quality)           14.58     0.29   20.88     0.42   14.87   21.30
Dutch (low/medium quality)      6.13     0.13    6.13     0.13    6.26    6.26
English (high quality)         48.53     0.99   48.53     0.98   49.52   49.51
English (low/medium quality)  155.22     3.17  155.22     3.17  158.39  158.39
New (high quality)              0.00     0.00   24.44     0.50    0.00   24.94
New (low/medium quality)        0.00     0.00    8.33     0.17    0.00    8.50
Other languages                47.37     0.97   47.37     0.97   48.34   48.34
Total                         355.77     7.24  538.23    10.97  363.01  549.20

In [6]:
# Create a simplified comparison table with only total counts for each epoch
df_comparison = df_formatted[[("epoch_1", "total"), ("epoch_2", "total")]].copy()
df_comparison.columns = ["Epoch 1 Total", "Epoch 2 Total"]

# Remove the 'Total' row to avoid redundancy
df_comparison = df_comparison.drop("Total")

# Calculate percentages for each epoch
epoch_1_total = df_comparison["Epoch 1 Total"].sum()
epoch_2_total = df_comparison["Epoch 2 Total"].sum()

df_comparison["Epoch 1 %"] = (
    df_comparison["Epoch 1 Total"] / epoch_1_total * 100
).round(2)
df_comparison["Epoch 2 %"] = (
    df_comparison["Epoch 2 Total"] / epoch_2_total * 100
).round(2)

# Export to markdown
comparison_markdown = df_comparison.to_markdown()

# Save to file
with open("token_counts.md", "w") as f:
    f.write("# Token Count Comparison: Epoch 1 vs Epoch 2\n\n")
    f.write("Token counts in billions for each data bucket with percentages:\n\n")
    f.write(comparison_markdown)

print("Epoch comparison table with percentages:")
print(df_comparison)
print("\nMarkdown saved to 'token_counts.md'")

Epoch comparison table with percentages:
                              Epoch 1 Total  Epoch 2 Total  Epoch 1 %  \
bucket                                                                  
Code                                  85.63         231.95      23.59   
Dutch (high quality)                  14.87          21.30       4.10   
Dutch (low/medium quality)             6.26           6.26       1.72   
English (high quality)                49.52          49.51      13.64   
English (low/medium quality)         158.39         158.39      43.63   
New (high quality)                     0.00          24.94       0.00   
New (low/medium quality)               0.00           8.50       0.00   
Other languages                       48.34          48.34      13.32   

                              Epoch 2 %  
bucket                                   
Code                              42.23  
Dutch (high quality)               3.88  
Dutch (low/medium quality)         1.14  
English (high qual

# Count tokens per dataset (estimated)

227309190811

In [20]:
import json

json_file_path = (
    os.path.join(os.environ["GPTNL_DATALOADER"], "count_tokens/counts_final_all.json")
)

with open(json_file_path, "r") as fp:
    counts_dict = json.load(fp)

counts_dict["train"]["code"] = {}
counts_dict["validate"]["code"] = {}
counts_dict["train"]["code"]["cc_github_open_source"] = epoch_2_split_counts["train"][
    "code"
]
counts_dict["validate"]["code"]["cc_github_open_source"] = epoch_2_split_counts[
    "validate"
]["code"]

# Unpack the nested dictionary into a list of records
records = []

for split, buckets in counts_dict.items():  # 'train', 'validate'
    for (
        bucket,
        datasets,
    ) in buckets.items():  # 'other_languages', 'new_low_medium_quality', etc.
        for dataset, count in datasets.items():  # 'Common Corpus', 'MultiEURLEX', etc.
            records.append(
                {
                    "Train/Valid": split,
                    "Bucket": bucket,
                    "Dataset": dataset,
                    "Count": count,
                }
            )

# Create DataFrame
df = pd.DataFrame(records)

# Optional: Sort by Train/Valid, then Bucket, then Dataset for better organization
df = df.sort_values(["Train/Valid", "Bucket", "Dataset"]).reset_index(drop=True)

print(f"DataFrame shape: {df.shape}")
print("\nFirst few rows:")
display(df.head(10))

print("\nUnique values in each column:")
print(f"Train/Valid: {df['Train/Valid'].unique()}")
print(f"Buckets: {df['Bucket'].unique()}")
print(f"Number of unique datasets: {df['Dataset'].nunique()}")

# Display some summary statistics
print("\nCount statistics:")
display(df["Count"].describe())
df.to_csv("counts_final_all.csv", index=False)

DataFrame shape: (2373, 4)

First few rows:


,Train/Valid,Bucket,Dataset,Count
0,train,code,cc_github_open_source,2.273092e+11
1,train,dutch_high_quality,Common Corpus,6.758306e+08
2,train,dutch_high_quality,"Dienst, Publiek en Communicatie",7.162692e+07
3,train,dutch_high_quality,European Parliament,4.630323e+07
4,train,dutch_high_quality,MultiEURLEX,8.418609e+07
5,train,dutch_high_quality,Naturalis,1.694926e+07
6,train,dutch_high_quality,Officiele Bekendmakingen,2.730992e+09
7,train,dutch_high_quality,Open Raadsinformatie,1.386420e+10
8,train,dutch_high_quality,Planbureau voor de Leefomgeving,1.417142e+07
9,train,dutch_high_quality,Rechtspraak,2.090362e+09



Unique values in each column:
Train/Valid: ['train' 'validate']
Buckets: ['code' 'dutch_high_quality' 'dutch_low_medium_quality'
 'english_high_quality' 'english_low_medium_quality' 'new_high_quality'
 'new_low_medium_quality' 'other_languages']
Number of unique datasets: 1190

Count statistics:


count    2.373000e+03
mean     2.314365e+08
std      5.647520e+09
min      2.800000e+01
25%      3.875800e+04
50%      2.182490e+05
75%      3.029289e+06
max      2.273092e+11
Name: Count, dtype: float64

In [27]:
dataset_counts = df.groupby(["Dataset"]).sum()["Count"]
total_tokens = dataset_counts.sum()
print(f"\nTotal tokens across all datasets: {total_tokens / 1e9:.2f} billion")

# Convert to DataFrame for better formatting
dataset_df = dataset_counts.reset_index()
dataset_df.columns = ["Dataset", "Token Count"]

# Sort by token count descending
dataset_df = dataset_df.sort_values("Token Count", ascending=False)

# Convert token count to billions and format
dataset_df["Token Count (Billions)"] = (dataset_df["Token Count"] / 1e9).round(4)

# Create markdown table
markdown_table = dataset_df[["Dataset", "Token Count (Billions)"]].to_markdown(
    index=False
)

# Save to file
with open("dataset_token_counts_epoch_2.md", "w") as f:
    f.write("# Dataset Token Counts (Epoch 2)\n\n")
    f.write(f"Total tokens across all datasets: {total_tokens / 1e9:.2f} billion\n\n")
    f.write("Token counts by dataset (in billions):\n\n")
    f.write(markdown_table)

print("Dataset counts saved to 'dataset_token_counts_epoch_2.md'")


Total tokens across all datasets: 549.20 billion
Dataset counts saved to 'dataset_token_counts_epoch_2.md'


In [5]:
import json

with open(
    os.path.join(os.environ["GPTNL_DATALOADER"], "count_tokens/counts_final_all.json"), "r"
) as fp:
    counts = json.load(fp)

print(counts)